# Power BI REST API Demo Notebook\n\nThis notebook is the primary demo entry point for the showcase. It presents the same reusable backend scripts through a guided, low-friction notebook experience so business users and developers can follow the flow together.\n\n## What this demo covers\n- Select an authentication mode\n- List Power BI workspaces\n- List datasets in a workspace\n- Execute a DAX query through the REST API\n- Review the trade-offs between service principal and delegated user authentication\n

## Prerequisites\n1. Install packages from `requirements.txt`.\n2. Copy `config/.env.example` to `.env` and fill in tenant, app, and workspace details.\n3. Make sure the Power BI tenant settings and dataset permissions required for your chosen auth mode are already configured.\n4. For delegated auth, the simplest demo path is device code flow.\n

In [ ]:
import os\nimport sys\nfrom pathlib import Path\n\nimport pandas as pd\nimport matplotlib.pyplot as plt\n\nrepo_root = Path.cwd().resolve().parent\nscripts_path = repo_root / 'scripts'\nif str(scripts_path) not in sys.path:\n    sys.path.append(str(scripts_path))\n\nfrom config_loader import load_config\nfrom list_workspaces import list_workspaces\nfrom list_datasets import list_datasets\nfrom execute_dax_query import execute_dax_query, DEFAULT_QUERY\n\npd.set_option('display.max_columns', 20)\npd.set_option('display.width', 120)\n

## Step 1 — Load configuration\nThe scripts and notebook use the same configuration model so the notebook stays simple while the code remains reusable.\n

In [ ]:
config = load_config()\nconfig

## Step 2 — Choose authentication mode\nUse `service_principal` for app-only automation. Use `delegated_user` when you want the notebook to run as the signed-in user context.\n

In [ ]:
AUTH_MODE = os.getenv('PBI_AUTH_MODE', 'service_principal')\nAUTH_MODE

## Step 3 — List workspaces\nThis step shows the first visible difference between auth modes: the workspace list reflects the access context of the caller.\n

In [ ]:
workspaces_df = list_workspaces(auth_mode=AUTH_MODE, top=50)\nworkspaces_df

## Step 4 — Pick a workspace\nYou can either set `PBI_WORKSPACE_ID` in `.env` or select the first workspace for a quick demo.\n

In [ ]:
workspace_id = config.workspace_id or (workspaces_df.iloc[0]['id'] if not workspaces_df.empty else None)\nworkspace_id

## Step 5 — List datasets\nThis step helps the audience see that the notebook is just a friendly interface over standard REST operations.\n

In [ ]:
datasets_df = list_datasets(workspace_id=workspace_id, auth_mode=AUTH_MODE)\ndatasets_df

## Step 6 — Pick a dataset and define a DAX query\nThe default query returns a compact regional summary that is easy to read on screen during a live presentation.\n

In [ ]:
dataset_id = config.dataset_id or (datasets_df.iloc[0]['id'] if not datasets_df.empty else None)\ndax_query = DEFAULT_QUERY\nprint(dax_query)

## Step 7 — Execute the DAX query\nIf your dataset uses RLS and you are demonstrating delegated auth, the results should align with the signed-in user's access. For a dual-auth demo with the exact same notebook flow, keep the main demo dataset free of RLS.\n

In [ ]:
query_df = execute_dax_query(\n    workspace_id=workspace_id,\n    dataset_id=dataset_id,\n    dax_query=dax_query,\n    auth_mode=AUTH_MODE,\n)\nquery_df

## Step 8 — Optional chart\nA simple bar chart makes the output feel more like a polished demo and less like a raw API response.\n

In [ ]:
if not query_df.empty and 'Dim Region[Region]' in query_df.columns and '[Total Sales]' in query_df.columns:\n    ax = query_df.plot(kind='bar', x='Dim Region[Region]', y='[Total Sales]', legend=False, figsize=(8, 4))\n    ax.set_title('Total Sales by Region')\n    ax.set_xlabel('Region')\n    ax.set_ylabel('Total Sales')\n    plt.xticks(rotation=0)\n    plt.tight_layout()\n    plt.show()\nelse:\n    print('Expected columns not found. Displaying table only.')\n

## Final summary\n- The notebook is the presentation-friendly entry point.\n- The scripts remain reusable for technical teams and CI/CD scenarios.\n- Service principal is best for unattended automation.\n- Delegated auth is best when you want the run to reflect an actual user context.\n